<img src="https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/banner-header-s2.svg" alt="Session 2 - Multi-Agent Systems" width="100%">

# Session 2 · Multi-Agent Systems

**Course: Agentic AI and Multi-Agent Systems** · Fundacion AI Granada
Tuesday 14 July 2026 · 10:00-12:00

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/montevive/agentic-ai-course/blob/main/session-2-multi-agent-systems.ipynb)

| Time | Block |
|---|---|
| 10:00 | From an agent to a team: coordination and delegation |
| 10:25 | Frameworks in depth: CrewAI vs LangGraph vs Microsoft Agent Framework |
| 10:50 | Workshop: implementing a multi-agent system |
| 11:30 | Efficient management of context, memory and RAG |

### What you'll learn

By the end of this session you will be able to:

1. Choose between sequential, hierarchical and consensual coordination, and argue the trade-offs (cost, quality, robustness).
2. Explain how agents communicate: message passing, agent-as-tool vs handoffs, supervisors, and where A2A fits.
3. Implement the same team in CrewAI (declarative roles) and LangGraph (state graph with conditional edges and checkpointing), and know when to use each.
4. Diagnose context growth and apply the four levers: pruning, compaction, offloading and budgets.
5. Build semantic RAG over a corpus with ChromaDB and tell it apart from an agent memory layer (Mem0, Letta, Zep).

Running the whole notebook costs well under 0.50 EUR in API calls with the default (cheap) models.

> **Secure & governed AI — the Montevive lens.** *[Montevive AI](https://montevive.ai) helps businesses apply AI securely and aligned with compliance (GDPR, NIS2, ISO 27001, AI Act). Today's angle: every handoff between agents copies data into another context, and every memory write is data retention.*

The **Solutions** section is at the end of the notebook.

## Setup

Run this cell first. It detects your environment (local or Colab), loads the API keys from `.env` (or Colab Secrets) and picks the working model based on the available provider.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q -r https://raw.githubusercontent.com/montevive/agentic-ai-course/main/requirements.txt
    if not Path("data").exists():
        !git clone --depth 1 https://github.com/montevive/agentic-ai-course.git /content/agentic-ai-course
        %cd /content/agentic-ai-course

from dotenv import load_dotenv
load_dotenv()

if IN_COLAB:
    from google.colab import userdata
    for key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"):
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass

PROVIDERS = {
    "anthropic": bool(os.getenv("ANTHROPIC_API_KEY")),
    "openai": bool(os.getenv("OPENAI_API_KEY")),
    "google": bool(os.getenv("GOOGLE_API_KEY")),
}

if PROVIDERS["anthropic"]:
    MODEL, MODEL_CHEAP = "anthropic:claude-sonnet-4-6", "anthropic:claude-haiku-4-5"
elif PROVIDERS["openai"]:
    MODEL, MODEL_CHEAP = "openai:gpt-5", "openai:gpt-5-mini"
elif PROVIDERS["google"]:
    MODEL, MODEL_CHEAP = "google-gla:gemini-pro-latest", "google-gla:gemini-flash-latest"
else:
    raise RuntimeError("No API keys. Check the 00-environment-check notebook.")

print("Available providers:", [k for k, v in PROVIDERS.items() if v])
print(f"Working model: {MODEL}")
print(f"Low-cost model: {MODEL_CHEAP}")

# Rule of thumb for agents: temperature 0 -> deterministic tool choice and
# structured output, reproducible runs. See Session 1, Block 2 for why.
AGENT_SETTINGS = {"temperature": 0.0}

We load the corpus and the term search from Session 1 (we'll use them as the teams' tools).

In [ ]:
import re

CORPUS = {p.stem: p.read_text(encoding="utf-8") for p in sorted(Path("data").glob("2*.md"))}

def search_corpus(query: str) -> str:
    terms = [t for t in re.findall(r"\w+", query.lower()) if len(t) > 3]
    scored = sorted(
        ((sum(text.lower().count(t) for t in terms), name) for name, text in CORPUS.items()),
        reverse=True,
    )
    top = [(s, n) for s, n in scored[:3] if s > 0]
    if not top:
        return "No results for that query."
    return "\n\n".join(f"[{name}]\n{CORPUS[name][:900]}" for _, name in top)

print(f"{len(CORPUS)} papers loaded from the corpus")

---

## 10:00 · Block 1: from an agent to a team

Why several agents instead of one with 30 tools?

- **Bounded context**: each agent sees only what it needs (fewer tokens, less confusion).
- **Specialization**: different instructions and tools per role; a critical reviewer works better if it isn't also the author.
- **Parallelism**: independent subtasks can run at the same time.

The price: more latency, more orchestration cost and new failure modes (two agents can agree... and be wrong).

### The three coordination patterns

![The three coordination patterns](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s2-coordination-patterns.svg)

| Pattern | Wins on | Use it when |
|---|---|---|
| Sequential | cost, predictability | the flow is known and linear (document ETL, report) |
| Hierarchical | quality, flexibility | the task requires deciding dynamically what to do |
| Consensual | robustness | an individual error is expensive (review, evaluation) |

*(The `2403-multiagent-coordination` paper in our corpus, though fictional, captures the real intuition well: hierarchical wins on quality, sequential on cost, and consensus only pays off when the individual agents fail a lot.)*

### Handoffs and messages between agents

Internally, communication between agents is **message passing**: the supervisor delegates by writing an instruction, the worker replies with an artifact.

> **Two mechanics for delegation.** In practice a "team" is wired one of two ways. **Agent-as-tool**: the supervisor keeps control and calls a specialist exactly like a tool; the result comes back as a tool result (Session 3's pipeline works this way). **Handoff**: the supervisor transfers the whole conversation to the specialist, which takes over from there (the OpenAI Agents SDK pattern). And a supervisor is not special infrastructure: it's just an agent whose tools happen to be other agents.

Between agents from **different organizations**, the emerging standard is **A2A (Agent2Agent)**: capability discovery via an *Agent Card*, task delegation and result verification over HTTP (spec and SDKs: [a2aproject/A2A](https://github.com/a2aproject/A2A)). An Agent Card snippet:

```json
{
  "name": "literature-review-agent",
  "description": "Literature review over biomedical corpora",
  "url": "https://agents.university.edu/litreview",
  "skills": [{"id": "search", "description": "Search and synthesis with citations"}],
  "capabilities": {"streaming": true}
}
```

The idea to keep: **inside** your system the frameworks handle coordination; **between** systems, A2A aims to be the HTTP of agents (we'll come back to it in Session 3, frontiers).

---

## 10:25 · Block 2: frameworks in depth

The same mini-task ("mini literature review": search the corpus -> synthesize -> critique) implemented in two frameworks with opposite philosophies.

### 2.1 · CrewAI: declarative roles

In CrewAI you describe **who each agent is** (role, goal, backstory) and **what tasks** there are, and the framework orchestrates. Maximum prototyping speed.

In [ ]:
from crewai import Agent as CrewAgent, Crew, Process, Task, LLM
from crewai.tools import tool as crew_tool

if PROVIDERS["anthropic"]:
    crew_llm = LLM(model="anthropic/claude-haiku-4-5", max_tokens=2048)
elif PROVIDERS["openai"]:
    crew_llm = LLM(model="openai/gpt-5-mini")
else:
    crew_llm = LLM(model="gemini/gemini-flash-latest")

@crew_tool("search_corpus")
def search_corpus_tool(query: str) -> str:
    """Search papers by terms in the local corpus of abstracts on agentic AI."""
    return search_corpus(query)

researcher = CrewAgent(
    role="Literature researcher",
    goal="Locate the corpus papers relevant to the question and extract their key data",
    backstory="A meticulous librarian: only states what is in the corpus and always cites the identifier.",
    tools=[search_corpus_tool],
    llm=crew_llm,
    verbose=True,
)

synthesizer = CrewAgent(
    role="Scientific synthesizer",
    goal="Write a short, precise, cited synthesis from the researcher's findings",
    backstory="An editor of systematic reviews: clarity, concrete data and no filler.",
    llm=crew_llm,
    verbose=True,
)

t1 = Task(
    description="Search the corpus for what is known about: {question}. Extract data and figures with their paper.",
    expected_output="A list of findings, each with its paper identifier and the concrete fact.",
    agent=researcher,
)
t2 = Task(
    description="With the previous findings, write a synthesis of at most 120 words with [paper] citations.",
    expected_output="Synthesis in English with citations in brackets.",
    agent=synthesizer,
    context=[t1],
)

crew = Crew(agents=[researcher, synthesizer], tasks=[t1, t2], process=Process.sequential)
crew_result = await crew.kickoff_async(inputs={"question": "agent memory versus vectorstores"})
print("\n=== RESULT ===\n")
print(crew_result.raw)

What CrewAI gives you for free: delegation, context passing between tasks (`context=[t1]`) and readable traces. What it takes away: fine-grained control of the flow. If you need loops, conditions or selective retries, that's the second approach.

### 2.2 · LangGraph: the team as a state graph

LangGraph models the system as a **graph**: nodes that transform a shared state and edges (even conditional ones) that decide the path. Full control, durability and *checkpointing*.

![The review pipeline as a state graph](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s2-review-graph.svg)

Note the shape: this is the *evaluator-optimizer* workflow from Session 1's "when NOT to build an agent" box. The flow is fixed; only the retry decision is made at runtime. That's exactly what makes it predictable enough for production.

In [ ]:
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

if PROVIDERS["anthropic"]:
    lg_llm = init_chat_model("claude-haiku-4-5", model_provider="anthropic")
elif PROVIDERS["openai"]:
    lg_llm = init_chat_model("gpt-5-mini", model_provider="openai")
else:
    lg_llm = init_chat_model("gemini-flash-latest", model_provider="google_genai")

class ReviewState(TypedDict):
    question: str
    findings: str
    synthesis: str
    verdict: str
    attempts: int

def research(state: ReviewState) -> dict:
    findings = search_corpus(state["question"])
    return {"findings": findings}

def synthesize(state: ReviewState) -> dict:
    prompt = (
        f"Question: {state['question']}\n\nCorpus findings:\n{state['findings']}\n\n"
        "Write a synthesis of at most 120 words in English, with [paper] citations."
    )
    if state.get("verdict", "").startswith("REJECTED"):
        prompt += f"\n\nThe previous version was rejected because: {state['verdict']}. Fix it."
    return {"synthesis": lg_llm.invoke(prompt).content, "attempts": state.get("attempts", 0) + 1}

def critique(state: ReviewState) -> dict:
    verdict = lg_llm.invoke(
        f"You are a strict reviewer. Synthesis:\n{state['synthesis']}\n\nFindings:\n{state['findings']}\n\n"
        "If every claim is supported by the findings and carries a citation, reply exactly 'APPROVED'. "
        "If not, reply 'REJECTED: ' followed by the reason on one line."
    ).content
    return {"verdict": verdict}

def decide(state: ReviewState) -> str:
    if state["verdict"].strip().startswith("APPROVED") or state["attempts"] >= 2:
        return "end"
    return "retry"

builder = StateGraph(ReviewState)
builder.add_node("research", research)
builder.add_node("synthesize", synthesize)
builder.add_node("critique", critique)
builder.add_edge(START, "research")
builder.add_edge("research", "synthesize")
builder.add_edge("synthesize", "critique")
builder.add_conditional_edges("critique", decide, {"end": END, "retry": "synthesize"})

graph = builder.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "review-001"}}
output = graph.invoke({"question": "context engineering and cost savings", "attempts": 0}, config)

print(output["synthesis"])
print("\nCritic verdict:", output["verdict"], f"(attempts: {output['attempts']})")

In [ ]:
snapshot = graph.get_state(config)
print("Persisted state for thread 'review-001':")
print("  keys:", list(snapshot.values.keys()))
print("  next pending node:", snapshot.next or "(none, finished)")

Notice the two things CrewAI didn't give you:

- **Conditional edge**: the critic can send the synthesis back to rewriting (with an attempt limit).
- **Checkpointing**: the state of each `thread_id` is persisted; you can resume an interrupted run, audit every step or do *time travel*. With a database checkpointer, this is production-grade durability. For research, the same mechanism gives you resumable, auditable, replayable runs: the multi-agent equivalent of a lab notebook (Session 3 turns this into full traceability).

### 2.3 · Microsoft Agent Framework (comparison, not run)

The evolution of AutoGen + Semantic Kernel, aimed at the .NET/Azure ecosystem. The same pipeline would be written like this:

```python
from agent_framework import SequentialBuilder
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(model_id="gpt-5-mini")
researcher = client.create_agent(instructions="Search and extract findings...", name="researcher")
synthesizer = client.create_agent(instructions="Synthesize with citations...", name="synthesizer")

workflow = SequentialBuilder().participants([researcher, synthesizer]).build()
```

Interesting if your institution lives in Azure (integrated identity, observability and deployment); a smaller community in the scientific-Python world than the previous two.

### 2.4 · Comparison table

| | CrewAI | LangGraph | MS Agent Framework | smolagents (S1) | NeMo Agent Toolkit (S3) |
|---|---|---|---|---|---|
| Paradigm | roles + tasks | state graph | teams + workflows | agent writes code | layer over other frameworks |
| Learning curve | low | medium-high | medium | low | medium |
| Flow control | limited | **full** (conditionals, loops) | medium | medium | inherits from the framework |
| Durability / checkpointing | basic | **native** | medium | no | no (adds profiling) |
| Observability | own traces | LangSmith and compatible | Azure Monitor | basic | **own profiler + eval** |
| Best for | quick team prototypes | production and reproducible research | Microsoft environments | compute tasks | profiling and evaluating pipelines |

Rule of thumb: **prototype with CrewAI, industrialize with LangGraph**, and profile/evaluate with cross-cutting tools (Session 3).

---

## 10:50 · Block 3: multi-agent workshop

### 🧪 Exercise: build your own team

Pick **one case** (or bring your own):

1. **Literature review**: researcher -> synthesizer -> critic that can reject (add the third role to the crew, or use the graph).
2. **Data analysis**: one agent proposes hypotheses about a dataset (simulated with a tool) and another verifies them.
3. **Teaching-content generation**: one agent writes 3 exam questions about a corpus paper and another solves them to validate that they are solvable.

Minimum requirements: 2-3 agents with distinct roles, at least one tool, and a quality-control mechanism (critic, verifier or conditional edge).

Working on simulations? The corpus paper `2406-abm-game-theory` sketches a fourth variant: heterogeneous LLM players with different strategies in a repeated game, coordinated as a graph. A good starting point if your research leans toward agent-based modeling.

Pick a framework: CrewAI (fast) or LangGraph (control). CrewAI skeleton:

In [ ]:
# 🧪 EXERCISE - CrewAI skeleton (example solution at the end of the notebook)

agent_1 = CrewAgent(
    role="TODO: role",
    goal="TODO: concrete, measurable goal",
    backstory="TODO: personality/criteria that shapes its behavior",
    tools=[search_corpus_tool],          # TODO: adjust or add your own tools
    llm=crew_llm,
    verbose=True,
)

agent_2 = CrewAgent(
    role="TODO: second role (e.g. critic/verifier)",
    goal="TODO",
    backstory="TODO",
    llm=crew_llm,
    verbose=True,
)

task_1 = Task(description="TODO: {input}", expected_output="TODO", agent=agent_1)
task_2 = Task(description="TODO", expected_output="TODO", agent=agent_2, context=[task_1])

# my_crew = Crew(agents=[agent_1, agent_2], tasks=[task_1, task_2], process=Process.sequential)
# print((await my_crew.kickoff_async(inputs={"input": "TODO"})).raw)

---

## 11:30 · Block 4: context, memory and RAG

### 4.1 · Context engineering: the window is a budget

Every turn resends the WHOLE history. Let's measure it:

In [ ]:
from pydantic_ai import Agent

meter = Agent(MODEL_CHEAP, model_settings=AGENT_SETTINGS, instructions="Answer in one sentence.")

history = None
for turn, question in enumerate([
    "What is an AI agent?",
    "And a multi-agent system?",
    "Give an example applied to biomedical research.",
], start=1):
    r = await meter.run(question, message_history=history)
    history = r.all_messages()
    usage = r.usage
    print(f"turn {turn}: input={usage.input_tokens:>5} tokens · output={usage.output_tokens:>4} tokens")

The input **grows turn by turn** even though the questions are equally short: you're paying for the past. With tool-using agents it's worse: every `tool_result` (sometimes thousands of tokens) travels forever. Strategies, from least to most aggressive:

| Strategy | What it does | When |
|---|---|---|
| **Pruning** | remove already-consumed tool results and irrelevant turns | always, cheap |
| **Compaction** | summarize the old history and replace it with the summary | ~70% window occupancy |
| **Offloading** | move facts to external memory (RAG/memory) and retrieve on demand | long-running agents |
| **Budgets** | explicit token/step limits per task | cost control in production |

![The context window is a budget: the four levers](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s2-context-budget.svg)

The impact is threefold: **cost** (input tokens are billed every turn, so an unmanaged history makes total conversation cost grow quadratically), **latency** (more input means a slower first token), and **reliability** (models attend worse to the middle of long contexts, the "lost in the middle" effect: [Liu et al., 2023](https://arxiv.org/abs/2307.03172)).

### 4.2 · Compaction in 10 lines

In [ ]:
summary = await meter.run(
    "Summarize this conversation in 2 sentences, keeping the key facts:\n\n"
    + "\n".join(str(m) for m in history)
)

compactor = Agent(
    MODEL_CHEAP, model_settings=AGENT_SETTINGS,
    instructions=f"Context from the previous conversation (summary): {summary.output}. Answer in one sentence.",
)
r = await compactor.run("Which applied example had you given me?")
print("With compacted context:", r.output)
print("input tokens:", r.usage.input_tokens, "(compare with turn 3 above)")

### 4.3 · RAG with ChromaDB: semantic search over the corpus

The term search from S1 fails if you ask with different words. RAG indexes by **meaning**: embeddings + vector database.

> **How RAG works.** An *embedding model* maps text to a vector so that semantically similar texts land close together. Indexing (once): embed every document and store the vectors. Query (every question): embed the question, retrieve the top-k nearest documents by similarity, and put them in the context ([Lewis et al., 2020](https://arxiv.org/abs/2005.11401)). Our 12-abstract corpus fits ChromaDB in memory; at scale the same pattern runs on FAISS, pgvector or Pinecone - only the store changes.

![The RAG pipeline: index once, retrieve per question](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s2-rag-pipeline.svg)

In [ ]:
import chromadb

chroma = chromadb.Client()
collection = chroma.get_or_create_collection("papers")

if collection.count() == 0:
    collection.add(
        ids=list(CORPUS.keys()),
        documents=list(CORPUS.values()),
        metadatas=[{"paper": k} for k in CORPUS],
    )
print(f"Collection with {collection.count()} indexed documents")

question = "how do I stop the agent from remembering stale information and overspending?"

semantic = collection.query(query_texts=[question], n_results=3)
print("\nSEMANTIC search:", semantic["ids"][0])
print("TERM search:", search_corpus(question)[:80], "...")

Semantic search finds the *context engineering* paper even though the question shares almost no vocabulary with it. Now, RAG **as an agent tool**:

In [ ]:
rag_agent = Agent(
    MODEL, model_settings=AGENT_SETTINGS,
    instructions=(
        "Research assistant. Use the semantic search tool and answer with [paper] citations. "
        "If the corpus doesn't cover the question, say so."
    ),
)

@rag_agent.tool_plain
def semantic_search(query: str) -> str:
    """Semantic search over the paper corpus; returns the 3 most relevant."""
    res = collection.query(query_texts=[query], n_results=3)
    return "\n\n".join(
        f"[{meta['paper']}]\n{doc[:600]}"
        for doc, meta in zip(res["documents"][0], res["metadatas"][0])
    )

r = await rag_agent.run("What practical heuristics are there to avoid saturating the context window?")
print(r.output)

### 4.4 · Agent memory layer vs vectorstore

A vectorstore **retrieves documents**; a memory layer (Mem0, Letta, Zep) **remembers facts**: it extracts, consolidates and updates salient information from conversations (user preferences, decisions made, previous results).

| | Vectorstore (RAG) | Memory layer |
|---|---|---|
| Unit | document chunk | consolidated fact ("Ana researches time series") |
| Writing | explicit indexing | automatic extraction from the conversation |
| Update | re-index | consolidates and **corrects** contradictory facts |
| Cost per query | medium (long chunks) | low (short facts) |
| Use it for | documentary knowledge | personalization and state across sessions |

> **Secure & governed AI — the Montevive lens.** *[Montevive AI](https://montevive.ai) helps businesses apply AI securely and aligned with compliance (GDPR, NIS2, ISO 27001, AI Act). Multi-agent data protection in two rules: **minimize what you hand off** (a specialist gets the fields it needs, not the whole conversation — cheaper, and GDPR art. 5), and **treat memory as retention**: consolidated facts about people are personal data, so the right to erasure (art. 17) must reach them — Mem0 exposes `delete()`/`delete_all(user_id=...)` for exactly that.*

Demo with Mem0 (uses an LLM to extract facts; requires `OPENAI_API_KEY` in its default configuration):

In [ ]:
if not PROVIDERS["openai"]:
    print("⚪ Optional demo: Mem0's default configuration uses OpenAI (LLM extractor + embeddings).")
else:
    try:
        from mem0 import Memory

        # Pin the extractor/embedder models explicitly: Mem0's defaults drift, and
        # newer OpenAI models reject the max_tokens parameter its defaults still send.
        memory = Memory.from_config({
            "llm": {"provider": "openai", "config": {"model": "gpt-4o-mini"}},
            "embedder": {"provider": "openai", "config": {"model": "text-embedding-3-small"}},
        })
        memory.add(
            [{"role": "user", "content": "I'm Ana, I research time series applied to energy "
                                          "and I prefer examples in Python with PyTorch."}],
            user_id="ana",
        )
        # Mem0 2.x scopes the search with filters=, not a user_id kwarg.
        found = memory.search("which framework does she prefer?", filters={"user_id": "ana"}, limit=5)
        for fact in found["results"]:
            print("-", fact["memory"])
    except Exception as exc:
        print(f"⚠️ Optional Mem0 demo not available in this environment: {type(exc).__name__}: {exc}")

---

## Takeaways

1. Multi-agent isn't an end in itself: it's the tool to **bound context and specialize**. A good single agent beats a bad team.
2. Sequential = cost, hierarchical = quality, consensus = robustness.
3. CrewAI to prototype, LangGraph to control (conditionals, checkpointing, resumption).
4. Context is a budget: prune, compact, offload (RAG/memory) and set limits.
5. RAG retrieves documents; agent memory remembers facts. They usually coexist.

**Next session (Thursday 16):** MCP, guardrails, traceability and an end-to-end multi-agent pipeline ready for production.

## Further reading

- [Lewis et al., 2020 - Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401): the original RAG paper.
- [Liu et al., 2023 - Lost in the Middle](https://arxiv.org/abs/2307.03172): why long contexts hurt reliability, not just cost.
- [Packer et al., 2023 - MemGPT](https://arxiv.org/abs/2310.08560): the operating-system view of agent memory (the idea behind Letta).
- [Anthropic - Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents): pruning, compaction and offloading in production.
- [A2A protocol](https://github.com/a2aproject/A2A): spec and SDKs for agent-to-agent collaboration.
- [LangGraph docs](https://langchain-ai.github.io/langgraph/) · [CrewAI docs](https://docs.crewai.com) · [Mem0 docs](https://docs.mem0.ai): the frameworks you used today.
- [Microsoft Agent Framework](https://github.com/microsoft/agent-framework): the .NET/Azure alternative from section 2.3.

---

## Solutions

### Workshop (case 1): literature review with a critic

In [ ]:
critic = CrewAgent(
    role="Critical reviewer",
    goal="Verify that every claim in the synthesis is supported by the corpus and carries a citation",
    backstory="A reviewer for a demanding journal: rejects unsupported claims, no exceptions.",
    tools=[search_corpus_tool],
    llm=crew_llm,
    verbose=True,
)

t3 = Task(
    description=(
        "Review the previous synthesis claim by claim using the search tool. "
        "Return the corrected final synthesis followed by a line 'VERDICT: ...'"
    ),
    expected_output="Final synthesis with citations and a verdict line.",
    agent=critic,
    context=[t2],
)

complete_crew = Crew(
    agents=[researcher, synthesizer, critic],
    tasks=[t1, t2, t3],
    process=Process.sequential,
)
result = await complete_crew.kickoff_async(inputs={"question": "guardrails and evaluation of agents in production"})
print(result.raw)

### Workshop (case 3, LangGraph variant): verified exam questions

Reuse the graph from block 2 by changing the nodes: `write_questions` (generates 3 questions about a paper) -> `solve` (another agent tries to answer them using only the paper) -> a conditional edge that sends the unsolvable questions back to writing. The structure is identical; you change the state (`questions`, `answers`, `verdict`) and the prompts.

---

<img src="https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/banner-footer.svg" alt="Montevive AI - montevive.ai - chema@montevive.ai" width="100%">

**Montevive AI** · [montevive.ai](https://montevive.ai) · chema@montevive.ai